# CS5788 Final Project: Prompt-to-Prompt on Real Images

This notebook runs the **DDIM inversion → optional null-text optimization → prompt-to-prompt** pipeline from the course writeup. Implementation lives in `ptp_utils.py`, `seq_aligner.py`, `p2p_controllers.py`, and `real_image_edit.py`.

**Inputs:** `IMG_8018.jpeg` and `headshot.jpg` are expected in the **same directory as this notebook** (the repo root). The setup cell resolves `IMAGE_PATH` / `PORTRAIT_PATH` from that folder so runs work even if the Jupyter working directory is not the project root.


In [ ]:
from pathlib import Path

import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler

import ptp_utils
import real_image_edit as re


def _repo_root() -> Path:
    """Directory containing this notebook and bundled assets."""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        nb = p / "cs5788_final_project.ipynb"
        if nb.is_file() and (p / "IMG_8018.jpeg").is_file() and (p / "headshot.jpg").is_file():
            return p
    for p in [cwd, *cwd.parents]:
        if (p / "cs5788_final_project.ipynb").is_file():
            return p
    return cwd


REPO_ROOT = _repo_root()
IMAGE_PATH = str(REPO_ROOT / "IMG_8018.jpeg")
PORTRAIT_PATH = str(REPO_ROOT / "headshot.jpg")

assert Path(IMAGE_PATH).is_file(), f"Missing cat photo: {IMAGE_PATH}"
assert Path(PORTRAIT_PATH).is_file(), f"Missing portrait: {PORTRAIT_PATH}"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float32,
).to(device)
model.scheduler = DDIMScheduler.from_config(model.scheduler.config)

re.init_project(model, num_ddim_steps=50, guidance_scale=7.5)

## 1. DDIM inversion: noising trajectory

Snapshots while noising the latent (inversion direction), for the source prompt used in the paper experiments.


In [ ]:
SOURCE_REALISTIC = "a realistic photo of a cat sitting next to a wall"

re.visualize_noising_process(
    model=model,
    image=IMAGE_PATH,
    source_prompt=SOURCE_REALISTIC,
    num_inference_steps=50,
    num_vis_steps=10,
)

## 2. Inversion-only edit (no null-text)

P2P with latents from DDIM inversion only—useful baseline from §3.1 of the writeup.


In [ ]:
SOURCE = "a realistic photo of a cat sitting next to a wall"
TARGET = "a watercolor painting of a cat sitting next to a wall"
prompts = [SOURCE, TARGET]

controller = re.AttentionReplace(
    prompts,
    re.get_runtime().num_ddim_steps,
    cross_replace_steps=0.4,
    self_replace_steps=0.9,
)

result_inv_only = re.edit_real_image(
    model=model,
    controller=controller,
    image=IMAGE_PATH,
    source_prompt=SOURCE,
    target_prompt=TARGET,
    num_inference_steps=50,
    guidance_scale=7.5,
)

ptp_utils.view_images(
    [result_inv_only["original"], result_inv_only["reconstruction"], result_inv_only["edited"]]
)

## 3. Null-text optimization (diagnostic grid)

Per-timestep **before / after** one denoising step using optimized null embeddings (see `visualize_null_text_optimization` in `real_image_edit.py`).


In [ ]:
re.visualize_null_text_optimization(
    model=model,
    image=IMAGE_PATH,
    source_prompt=SOURCE_REALISTIC,
    num_inference_steps=50,
    num_inner_steps=10,
    num_vis_steps=6,
)

## 4. Null-text + P2P: watercolor with local blend

Cross-attention blend on *watercolor* / *painting* tokens (Table 1 in the writeup).


In [ ]:
SOURCE = "a realistic photo of a cat sitting next to a wall"
TARGET = "a watercolor painting of a cat sitting next to a wall"

lb = re.LocalBlend(
    [SOURCE, TARGET],
    [[], ["watercolor", "painting"]],
)

controller = re.AttentionRefine(
    [SOURCE, TARGET],
    re.get_runtime().num_ddim_steps,
    cross_replace_steps=0.3,
    self_replace_steps=0.9,
    local_blend=lb,
)

result_watercolor = re.edit_real_image_null(
    model=model,
    controller=controller,
    image=IMAGE_PATH,
    source_prompt=SOURCE,
    target_prompt=TARGET,
    guidance_scale=7.5,
    num_inner_steps=10,
    early_stop_epsilon=1e-5,
)

ptp_utils.view_images(
    [result_watercolor["original"], result_watercolor["reconstruction"], result_watercolor["edited"]]
)

## 5. Null-text + P2P: subject change (cat → dog)

Higher null-text inner steps (20) as in the subject-edit discussion.


In [ ]:
SOURCE = "a cat sitting next to a wall"
TARGET = "a dog sitting next to a wall"

controller = re.AttentionRefine(
    [SOURCE, TARGET],
    re.get_runtime().num_ddim_steps,
    cross_replace_steps=0.4,
    self_replace_steps=0.8,
)

result_dog = re.edit_real_image_null(
    model=model,
    controller=controller,
    image=IMAGE_PATH,
    source_prompt=SOURCE,
    target_prompt=TARGET,
    guidance_scale=7.5,
    num_inner_steps=20,
    early_stop_epsilon=1e-5,
)

ptp_utils.view_images(
    [result_dog["original"], result_dog["reconstruction"], result_dog["edited"]]
)

## 6. Null-text + P2P: watercolor dog (same photo, dog in prompt)

Combined subject + style edit from the writeup: source prompt still describes the **cat** photo; target asks for a **watercolor dog** (Table 1 style: `num_inner_steps=20`).

In [ ]:
SOURCE = "a realistic photo of a cat sitting next to a wall"
TARGET = "a watercolor painting of a dog sitting next to a wall"

controller = re.AttentionRefine(
    [SOURCE, TARGET],
    re.get_runtime().num_ddim_steps,
    cross_replace_steps=0.4,
    self_replace_steps=0.8,
)

result_watercolor_dog = re.edit_real_image_null(
    model=model,
    controller=controller,
    image=IMAGE_PATH,
    source_prompt=SOURCE,
    target_prompt=TARGET,
    guidance_scale=7.5,
    num_inner_steps=20,
    early_stop_epsilon=1e-5,
    offsets=(0, 0, 0, 0),
    low_resource=False,
)

ptp_utils.view_images(
    [
        result_watercolor_dog["original"],
        result_watercolor_dog["reconstruction"],
        result_watercolor_dog["edited"],
    ]
)

## 7. Null-text + P2P: best anime cat

Strong anime-style target prompt with cross/self replace tuned for the cat photo (`num_inner_steps=15`).

In [ ]:
SOURCE = "a realistic photo of a cat sitting next to a wall"
TARGET = (
    "a cel-shaded, flat color, hand-drawn, clean outlines, 2D style anime illustration of a cat "
    "sitting next to a smooth indoor wall with a simplified background, focus on the cat, "
    "focus on the eyes, anime character design"
)

controller = re.AttentionRefine(
    [SOURCE, TARGET],
    re.get_runtime().num_ddim_steps,
    cross_replace_steps=0.45,
    self_replace_steps=0.55,
)

result_anime_cat = re.edit_real_image_null(
    model=model,
    controller=controller,
    image=IMAGE_PATH,
    source_prompt=SOURCE,
    target_prompt=TARGET,
    guidance_scale=7.5,
    num_inner_steps=15,
    early_stop_epsilon=1e-5,
    offsets=(0, 0, 0, 0),
    low_resource=False,
)

ptp_utils.view_images(
    [
        result_anime_cat["original"],
        result_anime_cat["reconstruction"],
        result_anime_cat["edited"],
    ]
)

## 8. Portrait → anime stylization (extended sweep)

Representative **strong prompt** + attention settings from `null_inversion_testing.ipynb` (requires `PORTRAIT_PATH`).


In [ ]:
SOURCE = "a realistic photo of a man with black hair"
TARGET = (
    "a cel-shaded, flat color, hand-drawn, clean outlines, 2D style anime illustration "
    "of a man with black hair, anime character design"
)

controller = re.AttentionRefine(
    [SOURCE, TARGET],
    re.get_runtime().num_ddim_steps,
    cross_replace_steps=0.45,
    self_replace_steps=0.55,
)

result_anime = re.edit_real_image_null(
    model=model,
    controller=controller,
    image=PORTRAIT_PATH,
    source_prompt=SOURCE,
    target_prompt=TARGET,
    guidance_scale=7.5,
    num_inner_steps=15,
    early_stop_epsilon=1e-5,
)

ptp_utils.view_images(
    [result_anime["original"], result_anime["reconstruction"], result_anime["edited"]]
)